# Common Neighbor Parameter (CNP)

The Common Neighbor Parameter ([Tsuzuki, Branicio & Rino, Comput. Phys. Commun. **177**, 518, 2007](https://doi.org/10.1016/j.cpc.2007.05.018)) is a per-atom scalar that quantifies deviation from a perfect crystal environment.

For atom $i$ with neighbors $j$:

$$\text{CNP}(i) = \frac{1}{n_i} \sum_j |\mathbf{Q}_{ij}|^2$$

where $\mathbf{Q}_{ij} = \sum_{k \in \text{common}(i,j)} (\mathbf{R}_{ik} + \mathbf{R}_{jk})$.

In a perfect crystal, symmetry forces every $\mathbf{Q}_{ij}$ to cancel, giving CNP = 0. Any defect — vacancy, surface, grain boundary, stacking fault — breaks this cancellation and produces CNP > 0.

Key advantage over CNA: CNP is a **continuous** scalar rather than a discrete label, so it can measure the *degree* of local disorder.

In [ ]:
import pyscal3
import numpy as np
from pyscal3.structures import make_crystal

## Perfect Crystals

FCC and BCC have CNP = 0 by symmetry. HCP has a small but non-zero CNP because the pair-wise $\mathbf{Q}_{ij}$ vectors do not fully cancel.

In [ ]:
structures = {
    "fcc": make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4)),
    "bcc": make_crystal("bcc", lattice_constant=2.87, repetitions=(4, 4, 4)),
    "hcp": make_crystal("hcp", lattice_constant=3.21, repetitions=(4, 4, 4)),
}

print("{:<10} {:>10} {:>10}".format("Structure", "CNP mean", "CNP max"))
print("-" * 32)

for name, atoms in structures.items():
    pyscal3.find_neighbors(atoms, method="cutoff", cutoff=0)
    cnp = pyscal3.common_neighbor_parameter(atoms)
    print(f"{name:<10} {cnp.mean():>10.4f} {cnp.max():>10.4f}")

## Detecting Thermal Disorder

Adding noise to a perfect crystal increases CNP, and the magnitude grows with the amount of displacement.

In [ ]:
print("{:<8} {:>10} {:>10}".format("Noise", "CNP mean", "CNP std"))
print("-" * 30)

for noise in [0.0, 0.02, 0.05, 0.1, 0.2]:
    atoms = make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4), noise=noise)
    pyscal3.find_neighbors(atoms, method="cutoff", cutoff=0)
    cnp = pyscal3.common_neighbor_parameter(atoms)
    print(f"{noise:<8.2f} {cnp.mean():>10.4f} {cnp.std():>10.4f}")

## FCC vs HCP Discrimination

Unlike CNA which assigns both FCC and HCP as “close-packed” environments with specific CNA signatures, CNP separates them quantitatively: FCC has CNP = 0, while HCP has CNP > 0.

In [ ]:
fcc = make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4))
hcp = make_crystal("hcp", lattice_constant=3.21, repetitions=(4, 4, 4))

for atoms in [fcc, hcp]:
    pyscal3.find_neighbors(atoms, method="cutoff", cutoff=0)
    pyscal3.common_neighbor_parameter(atoms)

print(f"FCC CNP: {fcc.arrays['pyscal_cnp'].mean():.6f}")
print(f"HCP CNP: {hcp.arrays['pyscal_cnp'].mean():.6f}")

## Accessing Results

CNP values are returned directly and stored in `atoms.arrays["pyscal_cnp"]`.

In [ ]:
atoms = make_crystal("fcc", lattice_constant=4.05, repetitions=(3, 3, 3), noise=0.05)
pyscal3.find_neighbors(atoms, method="cutoff", cutoff=0)
cnp = pyscal3.common_neighbor_parameter(atoms)

print(f"Returned array shape: {cnp.shape}")
print("Stored in atoms.arrays:", "pyscal_cnp" in atoms.arrays)
print(f"First 5 values: {cnp[:5]}")

## References

1. H. Tsuzuki, P. S. Branicio and J. P. Rino, *Structural characterization of deformed crystals by analysis of common atomic neighborhood*, Comput. Phys. Commun. **177**, 518 (2007). [doi:10.1016/j.cpc.2007.05.018](https://doi.org/10.1016/j.cpc.2007.05.018)